# كشف لوحات السيارات باستخدام YOLOv8

هذا الـ Notebook يقوم بـ:
1. تثبيت Ultralytics YOLO
2. تحميل نموذج جاهز ومدرّب مسبقاً على كشف لوحات السيارات
3. رفع صورك من جهازك
4. اختبار النموذج وعرض النتائج بصرياً
5. تنزيل النتائج (الصور مع المربعات المحيطة) كملف ZIP
6. (لاحقاً) Fine-tuning إذا احتجنا لتحسين الدقة

**قبل البدء:** تأكد من تفعيل GPU من القائمة:
`Runtime → Change runtime type → Hardware accelerator → T4 GPU`

## الخطوة 1: التحقق من تفعيل GPU

In [ ]:
!nvidia-smi

إذا ظهر لك جدول فيه معلومات الـ GPU (مثل Tesla T4) فأنت جاهز.

إذا ظهرت رسالة `command not found` → اذهب لـ Runtime → Change runtime type واختر GPU.

## الخطوة 2: تثبيت Ultralytics والمكتبات اللازمة

In [ ]:
!pip install -q ultralytics huggingface_hub

import ultralytics
ultralytics.checks()

## الخطوة 3: تحميل نموذج جاهز لكشف اللوحات

سنجرّب عدة نماذج جاهزة على Hugging Face بالترتيب، وأي واحد ينجح نستخدمه. كل هذه النماذج مدربة على كشف فئة واحدة فقط: `license_plate`.

In [ ]:
from huggingface_hub import hf_hub_download
from ultralytics import YOLO
import os

# قائمة بنماذج جاهزة - نجرّبها بالترتيب
candidates = [
    ("keremberke/yolov8m-license-plate", "best.pt"),
    ("keremberke/yolov8n-license-plate", "best.pt"),
    ("morsetechlab/yolov11-license-plate-detection", "yolov11n-license-plate-detection.pt"),
]

model = None
for repo_id, filename in candidates:
    try:
        print(f"محاولة تنزيل: {repo_id}")
        model_path = hf_hub_download(repo_id=repo_id, filename=filename)
        model = YOLO(model_path)
        print(f"✅ نجح التحميل من: {repo_id}\n")
        break
    except Exception as e:
        print(f"❌ فشل: {str(e)[:120]}\n")

assert model is not None, "فشلت كل النماذج - تحقق من الاتصال بالإنترنت"
print(f"معلومات النموذج:")
print(f"عدد الفئات: {len(model.names)}")
print(f"الفئات: {model.names}")

## الخطوة 4: رفع صورك للاختبار

شغّل الخلية التالية ثم اضغط على زر `Choose Files` لاختيار صورك (يمكنك اختيار عدة صور دفعة واحدة).

**الصيغ المدعومة:** JPG, JPEG, PNG, BMP, WEBP

In [ ]:
import os
import shutil
from google.colab import files

# إنشاء مجلد للصور المرفوعة
INPUT_DIR = "/content/test_images"
if os.path.exists(INPUT_DIR):
    shutil.rmtree(INPUT_DIR)
os.makedirs(INPUT_DIR, exist_ok=True)

# رفع الصور
print("اختر الصور من جهازك...")
uploaded = files.upload()

# نقل الصور إلى المجلد
for filename in uploaded.keys():
    shutil.move(filename, os.path.join(INPUT_DIR, filename))

# عدّ الصور
image_files = [f for f in os.listdir(INPUT_DIR) 
               if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.webp'))]
print(f"\nتم رفع {len(image_files)} صورة إلى {INPUT_DIR}")

## الخطوة 5: اختبار النموذج على صورة واحدة (معاينة)

نختبر أولاً على صورة واحدة لنرى جودة الكشف.

In [ ]:
import matplotlib.pyplot as plt
import cv2
from pathlib import Path

# تحقق أن النموذج محمّل
assert 'model' in dir() and model is not None, "⚠️ شغّل الخطوة 3 (تحميل النموذج) أولاً"

# اختر أول صورة للاختبار
image_files = sorted(os.listdir(INPUT_DIR))
test_image = os.path.join(INPUT_DIR, image_files[0])
print(f"اختبار على: {image_files[0]}")

# تشغيل الكشف
# conf = حد الثقة (0.25 افتراضي)، يمكن تقليله للحصول على كشف أكثر أو رفعه للدقة الأعلى
results = model.predict(test_image, conf=0.25, iou=0.45, verbose=False)

# عرض النتيجة
result = results[0]
annotated_img = result.plot()  # ترسم المربعات تلقائياً
annotated_img_rgb = cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(14, 10))
plt.imshow(annotated_img_rgb)
plt.axis('off')
plt.title(f"عدد اللوحات المكتشفة: {len(result.boxes)}")
plt.show()

# طباعة تفاصيل المربعات
if len(result.boxes) > 0:
    print("\nتفاصيل المربعات المكتشفة:")
    for i, box in enumerate(result.boxes):
        xyxy = box.xyxy[0].cpu().numpy()
        conf = float(box.conf[0])
        print(f"  لوحة {i+1}: confidence={conf:.3f}, bbox=[x1={xyxy[0]:.0f}, y1={xyxy[1]:.0f}, x2={xyxy[2]:.0f}, y2={xyxy[3]:.0f}]")
else:
    print("\n⚠️ لم يتم اكتشاف أي لوحة - جرب تقليل قيمة conf إلى 0.1")

## الخطوة 6: اختبار النموذج على كل الصور وحفظ النتائج

نمرّر كل الصور المرفوعة على النموذج، ونحفظ:
- الصور مع المربعات (للمراجعة البصرية)
- ملف CSV بكل المربعات المكتشفة
- اللوحات مقصوصة (crops) في مجلد منفصل - مفيد لـ OCR لاحقاً

In [ ]:
import csv
from tqdm import tqdm

OUTPUT_DIR = "/content/results"
ANNOTATED_DIR = os.path.join(OUTPUT_DIR, "annotated")
CROPS_DIR = os.path.join(OUTPUT_DIR, "crops")

# إعادة إنشاء المجلدات
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
os.makedirs(ANNOTATED_DIR, exist_ok=True)
os.makedirs(CROPS_DIR, exist_ok=True)

# إعدادات الكشف - يمكنك تعديلها
CONFIDENCE_THRESHOLD = 0.25  # حد الثقة - قلّله لكشف أكثر، ارفعه لدقة أعلى
IOU_THRESHOLD = 0.45         # حد التداخل بين المربعات

# تشغيل الكشف على كل الصور
csv_rows = []
stats = {"total_images": 0, "images_with_plates": 0, "total_plates": 0, "images_no_detection": []}

for img_file in tqdm(sorted(os.listdir(INPUT_DIR)), desc="معالجة الصور"):
    if not img_file.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.webp')):
        continue
    
    img_path = os.path.join(INPUT_DIR, img_file)
    stats["total_images"] += 1
    
    # تشغيل الكشف
    results = model.predict(img_path, conf=CONFIDENCE_THRESHOLD, iou=IOU_THRESHOLD, verbose=False)
    result = results[0]
    
    # حفظ الصورة مع المربعات
    annotated = result.plot()
    cv2.imwrite(os.path.join(ANNOTATED_DIR, img_file), annotated)
    
    n_boxes = len(result.boxes)
    if n_boxes > 0:
        stats["images_with_plates"] += 1
        stats["total_plates"] += n_boxes
        
        # قراءة الصورة الأصلية لقص اللوحات
        original = cv2.imread(img_path)
        
        for i, box in enumerate(result.boxes):
            xyxy = box.xyxy[0].cpu().numpy().astype(int)
            conf = float(box.conf[0])
            
            # قص اللوحة
            x1, y1, x2, y2 = xyxy
            crop = original[y1:y2, x1:x2]
            crop_name = f"{Path(img_file).stem}_plate{i+1}_conf{conf:.2f}.jpg"
            if crop.size > 0:
                cv2.imwrite(os.path.join(CROPS_DIR, crop_name), crop)
            
            # إضافة للـ CSV
            csv_rows.append({
                "image": img_file,
                "plate_id": i + 1,
                "confidence": round(conf, 4),
                "x1": int(x1), "y1": int(y1),
                "x2": int(x2), "y2": int(y2),
                "width": int(x2 - x1), "height": int(y2 - y1)
            })
    else:
        stats["images_no_detection"].append(img_file)

# حفظ CSV
csv_path = os.path.join(OUTPUT_DIR, "detections.csv")
with open(csv_path, "w", newline="", encoding="utf-8") as f:
    if csv_rows:
        writer = csv.DictWriter(f, fieldnames=csv_rows[0].keys())
        writer.writeheader()
        writer.writerows(csv_rows)

# طباعة الإحصائيات
print("\n" + "="*50)
print("الإحصائيات الإجمالية")
print("="*50)
print(f"إجمالي الصور: {stats['total_images']}")
print(f"صور تم اكتشاف لوحات فيها: {stats['images_with_plates']} ({100*stats['images_with_plates']/max(stats['total_images'],1):.1f}%)")
print(f"صور بدون اكتشاف: {len(stats['images_no_detection'])}")
print(f"إجمالي اللوحات المكتشفة: {stats['total_plates']}")
if stats["images_no_detection"]:
    print(f"\nالصور بدون كشف:")
    for img in stats["images_no_detection"][:10]:
        print(f"  - {img}")
    if len(stats["images_no_detection"]) > 10:
        print(f"  ... و {len(stats['images_no_detection'])-10} أخرى")

## الخطوة 7: عرض شبكة من النتائج للمراجعة السريعة

In [ ]:
import math

# اعرض أول 12 صورة مع المربعات
annotated_files = sorted(os.listdir(ANNOTATED_DIR))[:12]
n = len(annotated_files)

if n > 0:
    cols = 3
    rows = math.ceil(n / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(16, 5*rows))
    axes = axes.flatten() if rows > 1 else [axes] if cols == 1 else axes
    
    for i, fname in enumerate(annotated_files):
        img = cv2.imread(os.path.join(ANNOTATED_DIR, fname))
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        axes[i].imshow(img_rgb)
        axes[i].set_title(fname, fontsize=9)
        axes[i].axis('off')
    
    for j in range(n, len(axes)):
        axes[j].axis('off')
    
    plt.tight_layout()
    plt.show()
else:
    print("لا توجد صور للعرض")

## الخطوة 8: تنزيل النتائج كملف ZIP

In [ ]:
# ضغط النتائج
zip_path = "/content/license_plate_results.zip"
shutil.make_archive("/content/license_plate_results", 'zip', OUTPUT_DIR)

# تنزيل تلقائي
files.download(zip_path)
print("تم التنزيل! الملف يحتوي على:")
print("  - مجلد annotated/: الصور مع المربعات")
print("  - مجلد crops/: اللوحات مقصوصة (لاستخدامها مع OCR)")
print("  - ملف detections.csv: إحداثيات كل المربعات")

## تقييم الأداء

**راجع الصور في `annotated/`** وقيّم:

1. **التغطية (Recall):** كم لوحة من أصل اللوحات الموجودة فعلاً تم اكتشافها؟
2. **الدقة (Precision):** كم من المربعات المرسومة هي لوحات حقيقية وليست أخطاء؟
3. **جودة الحدود:** هل المربعات محاذية جيداً للوحة أم متوسعة/متضيقة؟

**إذا كانت النتيجة:**
- ≥ 90% صحيحة → النموذج جيد، يمكنك استخدامه مباشرة
- 70-90% → يحتاج fine-tuning خفيف
- < 70% → يحتاج fine-tuning كامل بصور من بيئتك

**تجارب يمكنك إجراؤها:**
- جرّب تقليل `CONFIDENCE_THRESHOLD` إلى 0.15 إذا كان النموذج يفوّت لوحات
- جرّب رفعه إلى 0.4 إذا كان يخطئ بكثرة (false positives)

---

# (لاحقاً) Fine-Tuning النموذج على بياناتك

إذا قررنا نحتاج تحسين النموذج، سنضيف هنا خلايا لـ:
1. تنزيل/توسيم بياناتك (Roboflow أو CVAT أو LabelImg)
2. تجهيز الـ dataset بصيغة YOLO
3. تشغيل التدريب
4. تقييم النموذج المحسّن (mAP, precision, recall)
5. مقارنة مع النموذج الأصلي

**لا تشغّل هذا القسم الآن** — أنتظر منك مراجعة نتائج الكشف الأولية أولاً.